In [1]:
%pip install nba_api

  Using cached nba_api-1.9.0-py3-none-any.whl (284 kB)
  Using cached pandas-2.2.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.1 MB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.28.1
    Uninstalling requests-2.28.1:
      Successfully uninstalled requests-2.28.1
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.4
    Uninstalling numpy-1.23.4:
      Successfully uninstalled numpy-1.23.4
  Attempting uninstall: pandas
    Found existing installation: pandas 1.5.1
    Uninstalling pandas-1.5.1:
      Successfully uninstalled pandas-1.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflic

In [2]:
%pip install --upgrade pandas


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import os
from datetime import datetime, timedelta
from src.config import *
from src.utils import *
from src.nba_scraping import *
print(pd.__version__)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/conda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/conda/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/conda/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.10/site-packages/traitlets/config/application.py", line 982, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.10/site-packages/ipykernel/

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/conda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/conda/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/conda/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.10/site-packages/traitlets/config/application.py", line 982, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.10/site-packages/ipykernel/

AttributeError: _ARRAY_API not found

2.2.3


# -- Setup du run

In [4]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
season = ['2024-25'] # Mets la saison que tu veux
os.makedirs(DATA_LAST_GAMES_DIR, exist_ok=True)
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_DIR, exist_ok=True)

# -- 1. Download nouveaux matchs (saison courante)

In [5]:
matchs_output_dir = os.path.join(DATA_LAST_GAMES_DIR, season[0])

last_games_path = download_games_for_seasons(season, matchs_output_dir, run_timestamp)

Extraction saison 2024-25


Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON'],
      dtype='object')


# -- 2. Comparaison avec l'historique pour extraire les nouveaux

In [6]:
hist_games_path = get_latest_file(DATA_GAMES_DIR)
games_to_scrape = get_new_games(hist_games_path, last_games_path)



#merge historical games with new games and remove duplicates
historical_games = pd.read_csv(hist_games_path, low_memory=False, dtype={'GAME_ID': str})

print(historical_games.head(2))

all_games = pd.concat([historical_games, games_to_scrape], ignore_index=True)

print(all_games.head(2))

save_dataframe_to_csv(all_games, DATA_LAST_GAMES_MERGED_DIR, 'all_', run_timestamp)


12 nouveaux matchs à traiter
   SEASON_ID     TEAM_ID TEAM_ABBREVIATION        TEAM_NAME     GAME_ID  \
0      22000  1610612756               PHX     Phoenix Suns  0020000011   
1      22000  1610612765               DET  Detroit Pistons  0020000005   

    GAME_DATE    MATCHUP WL  MIN  PTS  ...  OREB  DREB  REB  AST  STL  BLK  \
0  2000-10-31  PHX @ GSW  L  240   94  ...    11    33   44   25   12    3   
1  2000-10-31  DET @ TOR  W  241  104  ...    11    34   45   21    7    6   

   TOV  PF  PLUS_MINUS   SEASON  
0   16  28        -2.0  2000-01  
1   12  27         9.0  2000-01  

[2 rows x 29 columns]
   SEASON_ID     TEAM_ID TEAM_ABBREVIATION        TEAM_NAME     GAME_ID  \
0      22000  1610612756               PHX     Phoenix Suns  0020000011   
1      22000  1610612765               DET  Detroit Pistons  0020000005   

    GAME_DATE    MATCHUP WL  MIN  PTS  ...  OREB  DREB  REB  AST  STL  BLK  \
0  2000-10-31  PHX @ GSW  L  240   94  ...    11    33   44   25   12    3   
1  

'data/raw_last/games_merged/all__2025-05-26_11-29-32.csv'

# -- 3. Scrape boxscores pour ces matchs

In [7]:
scrape_boxscores_for_games(games_to_scrape, DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp, batch_size=BATCH_SIZE)

[1/12] GAME_ID: 0042400311 - 2025-05-20
[2/12] GAME_ID: 0042400311 - 2025-05-20
[3/12] GAME_ID: 0042400301 - 2025-05-21
[4/12] GAME_ID: 0042400301 - 2025-05-21
[5/12] GAME_ID: 0042400312 - 2025-05-22
[6/12] GAME_ID: 0042400312 - 2025-05-22
[7/12] GAME_ID: 0042400302 - 2025-05-23
[8/12] GAME_ID: 0042400302 - 2025-05-23
[9/12] GAME_ID: 0042400313 - 2025-05-24
[10/12] GAME_ID: 0042400313 - 2025-05-24
[11/12] GAME_ID: 0042400303 - 2025-05-25
[12/12] GAME_ID: 0042400303 - 2025-05-25
✅ Batch 1 saved with 354 rows


True

# -- 4. Merge historique + nouveaux (batchs)

In [8]:
merge_boxscores_batches(
    hist_dir=DATA_BOXSCORES_BATCHES_MERGED_DIR,
    new_dir=os.path.join(DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp),
    out_dir=DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, 
    run_timestamp=run_timestamp
)

Merged boxscores from data/raw/boxscores/batches_merged/boxscores_full__2025-05-23_17-32-44.csv and data/raw_last/batches/2025-05-26_11-29-32  saved at data/raw_last/batches_merged/merged_boxscores_2025-05-26_11-29-32.csv


'data/raw_last/batches_merged/merged_boxscores_2025-05-26_11-29-32.csv'